# Taller: Políticas de Gobernanza para Arquitectura IA-Ready (Privacidad y Sesgo)

Objetivo: diseñar políticas de gobierno de datos para una plataforma IA/Big Data que asegure privacidad, cumplimiento y control de sesgos/deriva de modelos. Evaluaremos el cumplimiento de productos de datos y modelos con una matriz de políticas.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Taller: Transición de Arquitectura Centralizada a Mesh/Fabric (Gov-Ready)](13_transicion_centralizado_a_mesh_fabric_taller_evaluacion.ipynb) | Siguiente: [Taller: Arquitectura IoT Industrial en Tiempo Real (Edge + Streaming + IA)](15_arquitectura_iot_industrial_streaming_edge_taller.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. 1. Ámbitos de política
2. 2. Plan de remediación y priorización
3. 3. Checklist de políticas IA-ready
4. 4. Métricas de Fairness y Deriva (Drift) Adicionales

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Taller: Políticas de Gobernanza para Arquitectura IA-Ready (Privacidad y Sesgo)"] --> C["Notebook siguiente"]
```


## 1. Ámbitos de política
- Privacidad y datos personales (PII/PHI/PCI)
- Acceso y control (RBAC/ABAC, mínimo privilegio)
- Ciclo de vida (retención/expurgo)
- Calidad/Observabilidad (SLOs, freshness, errores)
- Linaje y trazabilidad (auditoría)
- Responsable de modelo (MLOps): fairness, bias, drift, explainability

In [ ]:
from dataclasses import dataclass
from typing import Dict, List
import pandas as pd

@dataclass
class DataAsset:
    key: str
    contains_pii: bool
    retention_days: int
    access_model: str
    lineage_complete: bool
    freshness_slo_m: int
    observed: bool

@dataclass
class ModelAsset:
    key: str
    uses_protected_attr: bool
    fairness_metric: float
    fairness_min: float
    drift_monitoring: bool
    explainability: bool

policies = {
    'access': {'allowed_models': ['RBAC', 'ABAC']},
    'retention': {'max_days': 365},
    'observability': {'freshness_max_m': 60, 'observed_required': True},
    'lineage': {'complete_required': True},
    'fairness': {'min_ratio': 0.8, 'no_protected_attr': True},
    'drift': {'monitoring_required': True},
    'explain': {'explainability_required': True}
}
assets = [
    DataAsset('silver.orders_clean', True, 180, 'RBAC', True, 30, True),
    DataAsset('gold.churn_scoring', True, 400, 'ABAC', False, 90, False)
]
models = [
    ModelAsset('models.churn_v1', False, 0.86, 0.8, True, True),
    ModelAsset('models.credit_risk_v2', True, 0.75, 0.8, False, False)
]

issues: List[Dict] = []
for asset in assets:
    if asset.retention_days > policies['retention']['max_days']:
        issues.append({'asset': asset.key, 'policy': 'retention', 'issue': f'retention_days={asset.retention_days}'})
    if asset.access_model not in policies['access']['allowed_models']:
        issues.append({'asset': asset.key, 'policy': 'access', 'issue': f'access_model={asset.access_model}'})
    if policies['lineage']['complete_required'] and not asset.lineage_complete:
        issues.append({'asset': asset.key, 'policy': 'lineage', 'issue': 'lineage incomplete'})
    if asset.freshness_slo_m > policies['observability']['freshness_max_m']:
        issues.append({'asset': asset.key, 'policy': 'observability', 'issue': f'freshness_slo_m={asset.freshness_slo_m}'})
    if policies['observability']['observed_required'] and not asset.observed:
        issues.append({'asset': asset.key, 'policy': 'observability', 'issue': 'not observed'})

for model in models:
    if policies['fairness']['no_protected_attr'] and model.uses_protected_attr:
        issues.append({'asset': model.key, 'policy': 'fairness', 'issue': 'uses protected attribute'})
    if model.fairness_metric < policies['fairness']['min_ratio']:
        issues.append({'asset': model.key, 'policy': 'fairness', 'issue': f'fairness={model.fairness_metric}'})
    if policies['drift']['monitoring_required'] and not model.drift_monitoring:
        issues.append({'asset': model.key, 'policy': 'drift', 'issue': 'no drift monitoring'})
    if policies['explain']['explainability_required'] and not model.explainability:
        issues.append({'asset': model.key, 'policy': 'explain', 'issue': 'no explainability'})

pd.DataFrame(issues)


## 2. Plan de remediación y priorización
Ejemplos de acciones: reducir retención, completar lineage, instrumentar observabilidad, eliminar atributos protegidos, habilitar monitoreo de drift y explicabilidad.

Matriz de priorización sugerida: impacto en riesgo x esfuerzo.

## 3. Checklist de políticas IA-ready
- [ ] Clasificación de datos y PII identificada
- [ ] Retención conforme a normativa
- [ ] Acceso con RBAC/ABAC + mínimo privilegio
- [ ] Lineage visible y completo
- [ ] SLOs de freshness/errores con alertas
- [ ] Modelos sin atributos protegidos o con justificación y controles
- [ ] Fairness metrics ≥ umbral
- [ ] Monitoreo de drift
- [ ] Explicabilidad (post-hoc o intrínseca)
- [ ] Auditoría y DSAR (solicitudes de datos)

## 4. Métricas de Fairness y Deriva (Drift) Adicionales
Amplíamos evaluación incorporando:
- Disparate Impact (DI): ratio de tasa positiva grupo protegido vs no protegido.
- Equal Opportunity Difference (EOD): diferencia en true positive rate entre grupos.
- Detección de drift estadístico: prueba KS (Kolmogorov-Smirnov) sobre distribución de una feature.

Limitaciones: ejemplos sintéticos, no sustituyen auditorías completas ni análisis de impacto regulatorio.

In [ ]:
# Ejemplo sintético de fairness y drift
import numpy as np, pandas as pd
np.random.seed(123)

# Simulamos predicciones binarias y etiquetas por grupo (protected vs non-protected)
N=1000
prot = np.random.choice([0,1], size=N, p=[0.7,0.3])  # 1=protegido
# supongamos etiquetas reales y predicciones con leve sesgo
y_true = np.random.choice([0,1], size=N, p=[0.6,0.4])
# modelo da menos positivos al grupo protegido
y_pred = (np.random.rand(N) < (0.35 - 0.07*prot)).astype(int)

# Tasa positiva por grupo
rate_prot = y_pred[prot==1].mean() if (prot==1).any() else np.nan
rate_non = y_pred[prot==0].mean()
disparate_impact = rate_prot / rate_non if rate_non>0 else np.nan

# Equal Opportunity: TPR por grupo
from collections import defaultdict
def tpr(y_t, y_p, mask):
    pos = (y_t[mask]==1)
    if pos.sum()==0: return np.nan
    return (y_p[mask][pos]==1).mean()

eod = tpr(y_true,y_pred,prot==1) - tpr(y_true,y_pred,prot==0)

# Drift: KS test sobre una feature
try:
    from scipy.stats import ks_2samp
    feat_ref = np.random.normal(0,1,size=500)
    feat_new = np.random.normal(0.2,1.1,size=500)
    ks_stat, ks_p = ks_2samp(feat_ref, feat_new)
except Exception:
    ks_stat, ks_p = None, None

pd.Series({
    'disparate_impact': disparate_impact,
    'equal_opportunity_diff': eod,
    'ks_stat': ks_stat,
    'ks_p_value': ks_p
})

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- NIST Privacy Framework: https://www.nist.gov/privacy-framework
- DAMA International: https://www.dama.org/
- NIST AI Risk Management Framework: https://www.nist.gov/itl/ai-risk-management-framework
- Referencias del curso: ../04-recursos/referencias.md
